# The Dutch Childcare Benefits Scandal: When a Risk Score Meets a Zero-Tolerance Policy

In this notebook I look at the Dutch childcare benefits scandal, the *toeslagenaffaire*, as a fifth mechanism behind an outcome gap, picking this series back up after the facial recognition notebook closed out its first run of four.

The Tokyo Medical University notebook was a deliberate manual adjustment. The COMPAS notebook was an honest score meeting differing base rates. The Amazon notebook was a neutral feature acting as an accidental proxy. The facial recognition notebook was a representation gap from underrepresented training data. This one is different from all four: a system that put a protected attribute directly into the score, and then fed that score into a decision policy with no room for being wrong.

In this notebook, I will:

- Summarize what happened in the Netherlands and what investigations found
- Simulate a risk-scoring system that uses nationality as an explicit input, next to a rare, nationality-independent fraud rate
- Measure the flagging gap this produces, and how little of it turns out to be actual fraud
- Try the fix that worked in the Amazon notebook, dropping the explicit input, and see that it isn't enough on its own
- Find the second problem that fix leaves standing: a zero-tolerance policy that turns every flag, right or wrong, into the harshest possible outcome
- Simulate the fix for that second problem, and see why a good score still needs a decision policy that can absorb being wrong

## 1. Background: What Was Reported

Between roughly 2013 and 2019, the Dutch tax authority (Belastingdienst) used a self-learning risk-classification system to flag childcare benefit applications it considered likely fraudulent. Investigations by Dutch journalists, the Dutch Data Protection Authority, and a parliamentary inquiry found that the system used applicants' nationality as an explicit input, and that having a second nationality, alongside minor or purely administrative errors on a form, was treated as a meaningful risk signal.

Roughly 26,000 families were wrongly accused of fraud. Once flagged, families were typically ordered to repay their entire benefit, often years' worth, immediately and in full, and were branded as fraudsters, with little practical room to explain a mistake before the demand landed. Families lost homes and went into severe debt, and more than a thousand children are estimated to have been placed in out-of-home care during the crisis that followed. The Dutch government resigned over the scandal in January 2021, and the tax authority was later fined by the Dutch Data Protection Authority over its handling of nationality data.

## 2. Why This Is a Different Mechanism

Every earlier notebook in this series involved a score that treated a protected attribute unequally without naming it outright, or a threshold policy that could, in principle, be loosened. Here nationality was not implied by some correlated feature, it was a named input to the risk score, and that specific mistake is exactly what the Amazon notebook's fix, drop the proxy, is built to handle.

What that fix does not touch is the second design choice sitting underneath this one: what happens once someone is flagged. In this system, being flagged did not lead to a closer look, it led directly to the maximum possible consequence, full repayment demanded at once and a fraud accusation on the record, applied identically whether the flag turned out to be right or wrong. That second choice turns any imperfect classifier, and every classifier is imperfect, into a machine for producing wrongly ruined families, in a way that fixing the score alone cannot undo.

## 3. A Note on the Simulation Below

This notebook does not use real government data or the actual scoring model, which was never fully published. I represent each family with a nationality group, a hidden true fraud status, and a small count of administrative errors, and I build a simple weighted risk score by hand rather than training one, to keep the two mechanisms, the biased input and the zero-tolerance policy, easy to isolate and measure separately.

## 4. Simulating Families and a Fraud Rate That Does Not Depend on Nationality

I split families into a majority group with a single nationality and a minority group with a dual nationality, and give both groups the exact same true fraud rate, mirroring the real situation where actual fraud was rare and not meaningfully linked to nationality. I also give each family a count of minor administrative errors, made slightly more common in the dual-nationality group, standing in for the extra friction that paperwork in a second language or an unfamiliar system can add, a friction with nothing to do with fraud.

In [ ]:
import random

random.seed(7)

N_FAMILIES = 4000
DUAL_SHARE = 0.30
TRUE_FRAUD_RATE = 0.04


def make_families(n, dual_share, fraud_rate):
    """Returns a list of family dicts with a nationality group and a true fraud flag."""
    families = []
    for _ in range(n):
        group = "dual" if random.random() < dual_share else "dutch"
        true_fraud = random.random() < fraud_rate
        admin_errors = random.choices([0, 1, 2, 3], weights=[70, 20, 7, 3])[0]
        if group == "dual":
            admin_errors = min(admin_errors + random.choices([0, 1], weights=[70, 30])[0], 3)
        families.append({"group": group, "true_fraud": true_fraud, "admin_errors": admin_errors})
    return families


families = make_families(N_FAMILIES, DUAL_SHARE, TRUE_FRAUD_RATE)

## 5. Building a Risk Score That Explicitly Weights Nationality

The score below adds points for administrative errors and for a noisy underlying fraud signal, the kind of inputs a fraud model can defend on its face. Then it adds a flat penalty just for belonging to the dual-nationality group, mirroring how the real system used nationality directly, not through some indirect correlation but as a named input to the score itself.

In [ ]:
def risk_score(family, use_nationality=True):
    """Returns a risk score built from admin errors, a noisy fraud signal, and (optionally) nationality."""
    score = family["admin_errors"] * 8
    score += 40 if family["true_fraud"] else 0
    score += random.gauss(0, 10)
    if use_nationality and family["group"] == "dual":
        score += 25
    return score


for family in families:
    family["score"] = risk_score(family)

## 6. Measuring the Flagging Gap

I flag the top 10% of families by score for a full fraud investigation, the way a system under pressure to catch fraud might set its cutoff, and compare the flagging rate between the two nationality groups.

In [ ]:
def percentile(values, fraction):
    """Returns the value at a given percentile of a list."""
    ordered = sorted(values)
    index = int(fraction * len(ordered))
    return ordered[min(index, len(ordered) - 1)]


def flag_rate(families, group):
    """Returns the fraction of a group's families that got flagged."""
    subset = [f for f in families if f["group"] == group]
    flagged = [f for f in subset if f["flagged"]]
    return len(flagged) / len(subset)


scores = [f["score"] for f in families]
flag_threshold = percentile(scores, 0.90)

for family in families:
    family["flagged"] = family["score"] >= flag_threshold

print("Flag rate, dutch-only families:", round(flag_rate(families, "dutch"), 3))
print("Flag rate, dual-nationality families:", round(flag_rate(families, "dual"), 3))

## 7. Measuring Precision: How Many Flagged Families Actually Committed Fraud

A flagging gap only tells half the story. I also check precision, the share of each group's flagged families that were actually committing fraud, to see how much of this flagging is finding real fraud versus just falling on one group harder.

In [ ]:
def precision(families, group):
    """Returns the fraction of a group's flagged families that were actually committing fraud."""
    flagged = [f for f in families if f["group"] == group and f["flagged"]]
    if not flagged:
        return 0.0
    true_positive = [f for f in flagged if f["true_fraud"]]
    return len(true_positive) / len(flagged)


print("Precision among flagged, dutch-only:", round(precision(families, "dutch"), 3))
print("Precision among flagged, dual-nationality:", round(precision(families, "dual"), 3))

## 8. The Zero-Tolerance Policy: From Flag to Catastrophe

In the real system, a flag did not trigger a closer look, it triggered the maximum consequence directly: full repayment demanded at once and a fraud accusation on the family's record, applied the same way whether the flag was right or wrong. I count how many families in each group were flagged and were not actually committing fraud, the families this policy ran over regardless of guilt.

In [ ]:
def wrongly_harmed(families, group):
    """Returns the count of a group's families flagged and given the full penalty despite not being fraud."""
    return len([f for f in families if f["group"] == group and f["flagged"] and not f["true_fraud"]])


dutch_total = len([f for f in families if f["group"] == "dutch"])
dual_total = len([f for f in families if f["group"] == "dual"])

print("Wrongly harmed, dutch-only:", wrongly_harmed(families, "dutch"), "out of", dutch_total)
print("Wrongly harmed, dual-nationality:", wrongly_harmed(families, "dual"), "out of", dual_total)

## 9. Trying the Fix That Worked Before

In the Amazon notebook, dropping the one proxy feature fixed the gap outright. I try the same move here: rebuild the score with the nationality term removed entirely, and recheck the flagging rate for both groups.

In [ ]:
random.seed(7)
families_no_nat = make_families(N_FAMILIES, DUAL_SHARE, TRUE_FRAUD_RATE)

for family in families_no_nat:
    family["score"] = risk_score(family, use_nationality=False)

scores_no_nat = [f["score"] for f in families_no_nat]
flag_threshold_no_nat = percentile(scores_no_nat, 0.90)

for family in families_no_nat:
    family["flagged"] = family["score"] >= flag_threshold_no_nat

print("Flag rate, dutch-only, no nationality input:", round(flag_rate(families_no_nat, "dutch"), 3))
print("Flag rate, dual-nationality, no nationality input:", round(flag_rate(families_no_nat, "dual"), 3))